In [2]:
import numpy as np, pandas as pd, gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
from pathlib import Path

ROOT = Path("..")      # the data files sit one folder above Notebooks/

L = np.load(ROOT / "L_metres.npy"); site_dist = np.load(ROOT / "site_dist.npy")       # from notebook 03

sites = pd.read_csv(ROOT / "sites.csv")
d = pd.read_csv(ROOT / "demand_d_it.csv", index_col=0); d.columns = [int(c) for c in d.columns]

DV0 = d.values
nJ, nI = L.shape
J, I, T = range(nJ), range(nI), range(4)

CAP = {"S": 1, "M": 3, "L": 5}
FIX = {"EMS": {"S": 0.0, "M": 1.0, "L": 2.0}, "OTHER": {"S": 1.5, "M": 2.5, "L": 3.5}}

F = {(j, k): FIX["EMS" if sites.loc[j, "site_type"] == "EMS" else "OTHER"][k] for j in J for k in CAP}

is_ems = (sites["site_type"] == "EMS").values
is_fire = (sites["site_type"] == "FIREHOUSE").values
H = 6
R = np.array([1.000, 2.374, 2.942, 2.371])           # the report's R_t

def make_a(theta=8, speed=15, beta=0.15):
    kappa = 1 + beta * np.clip(R - 1, 0, None)
    miles = (L + site_dist[:, None]) / 1609.344
    tau = kappa[None, None, :] * miles[:, :, None] / speed * 60
    return tau <= theta

def reach(a, dv):                                   # share of demand that at least one site can reach
    return sum(dv[a[:, :, t].any(axis=0), t].sum() for t in T) / dv.sum()

def solve(a, dv, B=30.0, P=None, rho=0.60, s=1.5, c=0.590, target=None, time_limit=30, gap=0.001):
    pairs = [(i, j, t) for j in J for i in I for t in T if a[j, i, t] and dv[i, t] > 0]
    by_it, by_jt = defaultdict(list), defaultdict(list)
    for i, j, t in pairs:
        by_it[(i, t)].append(j); by_jt[(j, t)].append(i)
    m = gp.Model(); m.setParam("OutputFlag", 0); m.setParam("TimeLimit", time_limit); m.setParam("MIPGap", gap)
    q = m.addVars(J, list(CAP), vtype=GRB.BINARY); n = m.addVars(J, vtype=GRB.INTEGER, lb=0)
    f = m.addVars(pairs, lb=0.0)
    cost = gp.quicksum(F[j, k] * q[j, k] for j in J for k in CAP) + c * n.sum()
    m.addConstrs((q.sum(j, "*") <= 1 for j in J))
    m.addConstrs((n[j] >= q.sum(j, "*") for j in J))
    m.addConstrs((n[j] <= gp.quicksum(CAP[k] * q[j, k] for k in CAP) for j in J))
    for (i, t), js in by_it.items():
        m.addConstr(gp.quicksum(f[i, j, t] for j in js) <= dv[i, t])
    for (j, t), is_ in by_jt.items():
        m.addConstr(s * gp.quicksum(f[i, j, t] for i in is_) <= rho * H * n[j])
    if P is not None:
        m.addConstr(n.sum() <= P)
    if target is None:                                  # mode 1: best coverage within budget B
        m.addConstr(cost <= B); m.setObjective(f.sum(), GRB.MAXIMIZE)
    else:                                               # mode 2: cheapest plan reaching target coverage
        m.addConstr(f.sum() >= target * dv.sum()); m.setObjective(cost, GRB.MINIMIZE)
    m.optimize()
    if m.SolCount == 0:
        return dict(coverage=np.nan, cost=np.nan, ambulances=np.nan)
    return dict(coverage=sum(v.X for v in f.values()) / dv.sum(), cost=cost.getValue(),
                ambulances=int(round(n.sum().getValue())))

def metrics(a, dv, **kw):
    r1 = solve(a, dv, B=10, **kw)
    r2 = solve(a, dv, target=0.95, **kw)
    return dict(max_reach=round(reach(a, dv), 4), cov_at_10m=round(r1["coverage"], 4),
                cost_for_95=round(r2["cost"], 2), ambulances_for_95=r2["ambulances"])

base = metrics(make_a(), DV0)
print(base)

Set parameter Username
Set parameter LicenseID to value 2881115
Academic license - for non-commercial use only - expires 2027-11-04
{'max_reach': np.float64(0.9994), 'cov_at_10m': np.float64(0.993), 'cost_for_95': 6.81, 'ambulances_for_95': 9}


In [3]:
rows = []
for theta in [6, 7, 8, 9, 10, 12]:
    rows.append(dict(theta_min=theta, **metrics(make_a(theta=theta), DV0)))
pd.DataFrame(rows)

,theta_min,max_reach,cov_at_10m,cost_for_95,ambulances_for_95
0,6,0.9975,0.8632,14.26,14
1,7,0.9994,0.9642,8.90,10
2,8,0.9994,0.9930,6.81,9
3,9,0.9994,0.9991,5.04,6
4,10,0.9994,0.9991,4.45,5
5,12,1.0000,0.9992,2.36,4


In [4]:
def build(a, dv, rho=0.60, s=1.5, c=0.590):
    pairs = [(i, j, t) for j in J for i in I for t in T if a[j, i, t] and dv[i, t] > 0]
    by_it, by_jt = defaultdict(list), defaultdict(list)
    for i, j, t in pairs:
        by_it[(i, t)].append(j); by_jt[(j, t)].append(i)
    m = gp.Model(); m.setParam("OutputFlag", 0)
    q = m.addVars(J, list(CAP), vtype=GRB.BINARY); n = m.addVars(J, vtype=GRB.INTEGER, lb=0)
    f = m.addVars(pairs, lb=0.0)
    cost = gp.quicksum(F[j, k] * q[j, k] for j in J for k in CAP) + c * n.sum()
    m.addConstrs((q.sum(j, "*") <= 1 for j in J))
    m.addConstrs((n[j] >= q.sum(j, "*") for j in J))
    m.addConstrs((n[j] <= gp.quicksum(CAP[k] * q[j, k] for k in CAP) for j in J))
    for (i, t), js in by_it.items():
        m.addConstr(gp.quicksum(f[i, j, t] for j in js) <= dv[i, t])
    for (j, t), is_ in by_jt.items():
        m.addConstr(s * gp.quicksum(f[i, j, t] for i in is_) <= rho * H * n[j])
    opened = {j: q.sum(j, "*") for j in J}
    for (i, j, t) in pairs:
        m.addConstr(f[i, j, t] <= dv[i, t] * opened[j])
    bud = m.addConstr(cost <= 30.0)
    m.setObjective(f.sum(), GRB.MAXIMIZE)
    return m, bud, dv.sum()

def cov_at(m, bud, total, B, time_limit=20, gap=0.001):
    bud.RHS = B
    m.setParam("TimeLimit", time_limit); m.setParam("MIPGap", gap)
    m.optimize()
    cov = m.ObjVal / total if m.SolCount else 0.0
    return cov, (m.Status == GRB.OPTIMAL)

In [5]:
import time
a0 = make_a()
m, bud, total = build(a0, DV0)
for B in (6, 10):
    t0 = time.time()
    cov, ok = cov_at(m, bud, total, B, time_limit=30, gap=0.001)
    print(f"B={B}: coverage {cov:.4f}, proven optimal: {ok}, seconds: {time.time()-t0:.1f}", flush=True)

B=6: coverage 0.9207, proven optimal: True, seconds: 1.1
B=10: coverage 0.9930, proven optimal: True, seconds: 0.6


In [6]:
import time

def scenario(a, dv, target=0.95):
    m, bud, total = build(a, dv)
    non_opt = 0
    cov10, ok = cov_at(m, bud, total, 10, time_limit=60); non_opt += (not ok)
    cov30, ok = cov_at(m, bud, total, 30, time_limit=60); non_opt += (not ok)
    if cov30 < target:
        need = np.nan
    else:
        lo, hi = 0.0, 30.0
        while hi - lo > 0.05:
            mid = (lo + hi) / 2
            cv, ok = cov_at(m, bud, total, mid, time_limit=60); non_opt += (not ok)
            if cv >= target: hi = mid
            else: lo = mid
        need = hi
    return dict(cov_at_10m=round(cov10, 4), budget_for_95=round(need, 2), non_optimal=non_opt)

def grow_uniform(dv, g):
    return dv * (1 + g)

def grow_random(dv, g, seed):
    w = np.random.default_rng(seed).dirichlet(np.ones(dv.shape[0]))
    profile = dv / dv.sum(axis=1, keepdims=True)
    return dv + (g * dv.sum() * w)[:, None] * profile

a0 = make_a()
N_SEEDS = 5
rows = []
t_start = time.time()
for g in [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    t0 = time.time()
    u = scenario(a0, grow_uniform(DV0, g))
    rnd = [scenario(a0, grow_random(DV0, g, seed)) for seed in range(N_SEEDS)]
    rows.append(dict(growth=g,
        uniform_cov_10m=u["cov_at_10m"], uniform_budget_95=u["budget_for_95"],
        random_cov_10m_mean=np.mean([r["cov_at_10m"] for r in rnd]).round(4),
        random_cov_10m_min=np.min([r["cov_at_10m"] for r in rnd]).round(4),
        random_budget_95_mean=np.mean([r["budget_for_95"] for r in rnd]).round(2),
        random_budget_95_max=np.max([r["budget_for_95"] for r in rnd]).round(2),
        non_optimal_solves=u["non_optimal"] + sum(r["non_optimal"] for r in rnd)))
    print(f"done growth {g:.2f} ({time.time() - t0:.0f}s)", flush=True)
print("total minutes:", round((time.time() - t_start) / 60, 1))
pd.DataFrame(rows)

done growth 0.00 (65s)
done growth 0.05 (74s)
done growth 0.10 (71s)
done growth 0.15 (80s)
done growth 0.20 (79s)
done growth 0.25 (79s)
done growth 0.30 (80s)
total minutes: 8.8


,growth,uniform_cov_10m,uniform_budget_95,random_cov_10m_mean,random_cov_10m_min,random_budget_95_mean,random_budget_95_max,non_optimal_solves
0,0.00,0.993,6.83,0.9930,0.9930,6.83,6.83,0
1,0.05,0.993,6.83,0.9925,0.9920,6.83,6.83,0
2,0.10,0.993,6.83,0.9921,0.9910,6.83,6.83,0
3,0.15,0.993,6.83,0.9917,0.9902,6.83,6.83,0
4,0.20,0.993,6.83,0.9913,0.9894,7.18,7.41,0
5,0.25,0.993,6.83,0.9910,0.9887,7.18,7.41,0
6,0.30,0.993,6.83,0.9907,0.9880,7.18,7.41,0
